Before EGU 2026, I decided to re-make the passage of OPT operator derivation since we need to take care of staggered grids etc. which are not easy to handle with the current version

# Big change!!

pointsInSpace, pointsInTime should be real (before they were -1 but now it should be 2,3,4 ...)

In [1]:
using Pkg
using Metal

cd(@__DIR__)
Pkg.activate("../")
ParamFile = "../config/testparam.csv"  # maybe GeoPoints and planet1D should be fusioned

# batchGPU should be at this level (I have not made it as a module yet, since the choice of Metal/CUDA should be done in a manual way)
include("../src/batchFiles/batchGPU.jl")


include("../src/commonBatchs.jl")

include("../src/planet1D.jl")
include("../src/GeoPoints.jl")

using .commonBatchs, .planet1D, .GeoPoints

  Activating 

devs = Metal.devices() = Metal.MTL.MTLDeviceInstance[Metal.MTL.MTLDeviceInstance (object of type AGXG13XDevice)]

project at `~/Documents/Github/flexOPT`



→ Using Metal backend (1 device(s))
Selected backend type: MetalBackend


In [ ]:
# Since this is the developing version, I do not use the module from flexOPT.jl

In [2]:

    include("../src/batchFiles/batchSymbolics.jl")
    export x,y,z,t
    export ∂x,∂y,∂z,∂t
    export ∂x²,∂y²,∂z²,∂t² 
    export Num2Float64,usefulPartials,myCoeff,mySolvefor,mySimplify,integrateTaylorPolynomials



use some solid packages

In [3]:
include("../src/motorsOPT/others.jl")

BouncingCoordinates (generic function with 1 method)

In [4]:
    include("../src/motorsOPT/famousEquations.jl")

famousEquation (generic function with 15 methods)

input parameters

In [ ]:
famousEquationType="2DacousticTime"
Δnum = (1.0,1.0,1.0)

orderBtime=-1
orderBspace=-1
pointsInSpace=3
pointsInTime=3

# new parameters for interpolated Taylor expansion
# μ points should be distributed from y_min+offsetμFromExtremitiesInΔy*Δy to y_max-offsetμFromExtremitiesInΔy*Δy
pointsμInSpace = 2
pointsμInTime = 2
offsetμFromExtremitiesInΔyInSpace=1/2
offsetμFromExtremitiesInΔyInTime=1/2

WorderBspace=-1
WorderBtime=-1
supplementaryOrder=2

We also propose different ways of boundary treatment (ghosting: Neumann/Dirichlet; same number of points etc.)

Starting the OPT derivation

In [ ]:
concreteParametersForOPTConstruction = @strdict famousEquationType Δnum orderBtime orderBspace WorderBtime WorderBspace supplementaryOrder pointsInSpace pointsInTime 

TaylorOptions=(WorderBtime=WorderBtime,WorderBspace=WorderBspace,supplementaryOrder=supplementaryOrder,pointsμInSpace=pointsμInSpace,pointsμInTime=pointsμInTime,offsetμInΔyInSpace=offsetμFromExtremitiesInΔyInSpace,offsetμInΔyInTime=offsetμFromExtremitiesInΔyInTime)
trialFunctionsCharacteristics=(orderBtime=orderBtime,orderBspace=orderBspace,pointsInSpace=pointsInSpace,pointsInTime=pointsInTime)

In [ ]:
exprs,fields,vars,extexprs,extfields,extvars,coordinates,∂,∂²=famousEquations(famousEquationType)
timeMarching,NtypeofExpr,NtypeofMaterialVariables,NtypeofFields,Ndimension,pointsUsed=numbersOfTheExpression(exprs,fields,vars,coordinates,trialFunctionsCharacteristics,TaylorOptions)

In [ ]:
pointsUsed